# GSoC 2026 - Discovery of Hidden Symmetries and Conservation Laws

**Task 1**: Dataset Preparation (rotate MNIST by 30 deg steps) + VAE Training
**Task 2**: Supervised Symmetry Discovery (MLP learns rotation in latent space)
**Task 3a**: Unsupervised Symmetry Discovery - binary 0/1 (paper approach)
**Task 3b**: Unsupervised Symmetry Discovery - full 0-9 (original extension)

This notebook explicitly sets all hyperparameters natively before training/inferencing a step so reviewers can easily change them!

In [ ]:
# ================================
# Environment Setup Cell
# ================================

import os
import sys
import subprocess

# ---- 1. Ensure project root is in path ----
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# ---- 2. Install required packages (safe for notebook users) ----
def install_if_missing(package):
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

install_if_missing("gdown")

import gdown
import torch
import matplotlib.pyplot as plt
import numpy as np

# ---- 3. Import project modules ----
from src.utils import DEVICE, CKPT, FIGS, show_cached_figure
from src.dataset import load_data, make_loader
from src.models import VAE, RotationMLP, LatentClassifier, SymmetryGenerator
from src.train import (
    train_vae, encode_dataset, train_rotation_mlp,
    train_classifier, train_symmetry_generator
)
from src.evaluate import (
    visualise_supervised, visualise_unsupervised,
    plot_symmetry_paths, visualise_rotated_samples,
    visualise_latent_space, visualise_reconstructions,
    visualise_rotation_trajectories, visualise_reconstructions_full
)

# ---- 4. Download weights if not already present ----
def download_weights_if_needed(download_dir=CKPT):
    os.makedirs(download_dir, exist_ok=True)

    # Check if folder already has files
    if len(os.listdir(download_dir)) > 0:
        print(f"✅ Weights already present in {download_dir}, skipping download.")
        return

    print(f"Downloading weights to {download_dir}...")

    url = "https://drive.google.com/drive/folders/1BZoCcW8YzKFzHbMExrWTXE_I3ctyaidX"
    
    gdown.download_folder(
        url=url,
        output=str(download_dir),
        quiet=False,
        use_cookies=False
    )

    print("✅ Download complete.")

# Run it
download_weights_if_needed()

# ---- 5. Device info ----
print(f"Using device: {DEVICE}")

## Task 1 - Dataset Preparation & VAE Training

**Objective**: Rotate every MNIST sample in steps of 30 deg (12 orientations per image), build and train a VAE with a low-dimensional latent space.

### Hyperparameters
Modify any values here before running the VAE training!

In [ ]:
# --- Task 1 Hyperparameters ---
ROTATION_STEP = 30
LATENT_DIM = 2
VAE_EPOCHS = 100
VAE_LR = 1e-3
VAE_BETA_MAX = 1.0
VAE_BETA_WARMUP = 10
BATCH_SIZE = 128

In [ ]:
print("Loading and rotating MNIST (digits 0-1)...")
train_bin, test_bin = load_data(max_digit=1, tag="rot_bin")
tr_ld = make_loader(train_bin, bs=BATCH_SIZE)
te_ld = make_loader(test_bin, bs=BATCH_SIZE, sh=False)

print(f"Train size: {len(train_bin['images'])}, Test size: {len(test_bin['images'])}")

In [ ]:
visualise_rotated_samples(train_bin, test_bin)


In [ ]:
# Train / load the binary VAE (digits 0-1)
vae = train_vae(tr_ld, te_ld, latent_dim=LATENT_DIM,
                epochs=VAE_EPOCHS, lr=VAE_LR,
                beta_warmup=VAE_BETA_WARMUP, beta_max=VAE_BETA_MAX,
                name=f"vae_{LATENT_DIM}d")

z_train = encode_dataset(vae, tr_ld, f"z{LATENT_DIM}_tr")
z_test  = encode_dataset(vae, te_ld, f"z{LATENT_DIM}_te")

In [ ]:
visualise_latent_space(z_train, z_test)
visualise_reconstructions(vae, test_bin)


## Task 2 - Supervised Symmetry Discovery

**Objective**: Learn an MLP that maps z -> z' corresponding to the known 30 deg rotation.
Apply iteratively to trace a full orbit.

### Hyperparameters

In [ ]:
# --- Task 2 Hyperparameters ---
MLP_EPOCHS = 200
MLP_LR = 1e-3
MLP_BATCH_SIZE = 256
MLP_HIDDEN = 128

In [ ]:
rot_mlp = train_rotation_mlp(z_train, 
                             latent_dim=LATENT_DIM, 
                             epochs=MLP_EPOCHS,
                             lr=MLP_LR,
                             batch_size=MLP_BATCH_SIZE,
                             name=f"rot_mlp_{LATENT_DIM}d")

visualise_supervised(vae, rot_mlp, z_test)

In [ ]:
visualise_rotation_trajectories(rot_mlp, z_test)


## Task 3a - Unsupervised Symmetry Discovery (binary 0/1)

**Objective**: Discover transformations G(z) that preserve classifier logits while strictly moving.

### Hyperparameters

In [ ]:
# --- Task 3a Hyperparameters ---
CLF_EPOCHS = 200
CLF_LR = 1e-3

GEN_EPOCHS = 800
GEN_LR = 2e-3
GEN_EPS = 1e-4
GEN_NORM_WEIGHT = 1.0

In [ ]:
clf_bin = train_classifier(
    z_train["z"], z_train["y"], z_test["z"], z_test["y"],
    latent_dim=LATENT_DIM, num_classes=1, epochs=CLF_EPOCHS, lr=CLF_LR,
    name=f"oracle_{LATENT_DIM}d")

gen_bin = train_symmetry_generator(
    clf_bin, z_train, latent_dim=LATENT_DIM,
    epochs=GEN_EPOCHS, eps=GEN_EPS, norm_weight=GEN_NORM_WEIGHT, lr=GEN_LR,
    name=f"gen1_{LATENT_DIM}d")

visualise_unsupervised(vae, gen_bin, z_test, total_steps=25000, n_show=10,
                       title="Task 3a: Unsupervised symmetry (0-1)",
                       fig_name="task3a_unsupervised_symmetry_01.png")

plot_symmetry_paths(gen_bin, z_test, steps=25000, record_every=100,
                    title="Task 3a: Symmetry paths (binary 0/1)",
                    fig_name="task3a_symmetry_paths_01.png")

## Task 3b - Unsupervised Symmetry Discovery (full 0-9)

**Objective**: Extended to all 10 digits. We use a higher-dimensional latent space (8D).

### Hyperparameters

In [ ]:
# --- Task 3b Hyperparameters ---
LATENT_FULL = 8
VAE_EPOCHS_FULL = 150
VAE_BETA_FULL = 0.01

CLF_EPOCHS_FULL = 200
GEN_EPOCHS_FULL = 800
GEN_EPS_FULL = 1e-4
GEN_NORM_WEIGHT_FULL = 1.0

In [ ]:
from sklearn.decomposition import PCA

print("Loading full MNIST (0-9)...")
train_full, test_full = load_data(max_digit=None, tag="rot_full")
tr_full_ld = make_loader(train_full, bs=BATCH_SIZE)
te_full_ld = make_loader(test_full, bs=BATCH_SIZE, sh=False)

# Low β to preserve latent structure for downstream symmetry discovery
vae_full = train_vae(tr_full_ld, te_full_ld, latent_dim=LATENT_FULL,
                     epochs=VAE_EPOCHS_FULL, beta_max=VAE_BETA_FULL, 
                     name=f"vae_{LATENT_FULL}d_lb")

z_train_full = encode_dataset(vae_full, tr_full_ld, f"z{LATENT_FULL}_lb_tr")
z_test_full  = encode_dataset(vae_full, te_full_ld, f"z{LATENT_FULL}_lb_te")

clf_full = train_classifier(
    z_train_full["z"], z_train_full["y"],
    z_test_full["z"], z_test_full["y"],
    latent_dim=LATENT_FULL, num_classes=10, epochs=CLF_EPOCHS_FULL,
    name=f"clf10_{LATENT_FULL}d_lb")

gen_full = train_symmetry_generator(
    clf_full, z_train_full, latent_dim=LATENT_FULL,
    epochs=GEN_EPOCHS_FULL, eps=GEN_EPS_FULL, norm_weight=GEN_NORM_WEIGHT_FULL,
    name=f"gen2_{LATENT_FULL}d_lb")

In [ ]:
visualise_reconstructions_full(vae_full, test_full)


In [ ]:
visualise_unsupervised(vae_full, gen_full, z_test_full, total_steps=25000,
                       n_show=10, n_rows=5,
                       title="Task 3b: Unsupervised symmetry (0-9)",
                       fig_name="task3b_unsupervised_symmetry_09.png")

# Symmetry paths projected to PCA space
if show_cached_figure("task3b_symmetry_paths_pca_09.png",
                      title="Task 3b: Symmetry paths in PCA space (0-9)",
                      figsize=(10, 10)):
    pass
else:
    gen_full.eval()
    fig, ax = plt.subplots(figsize=(10, 10))
    # Fit PCA
    pca = PCA(n_components=2).fit(z_test_full["z"].numpy())
    z2d = pca.transform(z_test_full["z"].numpy())
    
    ax.scatter(z2d[:, 0], z2d[:, 1], c=z_test_full["y"].numpy(),
               s=0.5, alpha=0.15, cmap="tab10")

    tab10_colors = plt.cm.tab10(np.linspace(0, 1, 10))
    for lbl in range(10):
        mask = (z_test_full["y"] == lbl) & (z_test_full["angle"] == 0)
        idx = mask.nonzero(as_tuple=True)[0]
        if len(idx) == 0: continue
        
        current = z_test_full["z"][idx[0]].unsqueeze(0).to(DEVICE)
        raw_path = [current.cpu().numpy().squeeze()]
        for i in range(1, 25001):
            with torch.no_grad():
                current = gen_full(current)
            if i % 100 == 0:
                raw_path.append(current.cpu().numpy().squeeze())
        
        raw_path = np.array(raw_path)
        path_2d = pca.transform(raw_path)
        ax.plot(path_2d[:, 0], path_2d[:, 1], color=tab10_colors[lbl], linewidth=1, alpha=0.8)
        ax.plot(path_2d[0, 0], path_2d[0, 1], "o", color=tab10_colors[lbl], markersize=6)

    ax.set(title="Task 3b: Symmetry paths in PCA space (0-9)", xlabel="PC1", ylabel="PC2")
    plt.tight_layout()
    from src.utils import savefig_cached
    savefig_cached(fig, "task3b_symmetry_paths_pca_09.png")
    plt.show()